# Metrics layer demo

This notebook shows how an analyst queries the Databox semantic metrics
layer against the flagship mart
(`analytics.fct_species_environment_daily`). Three charts cover the three
consumption patterns:

1. A daily time-series of a single metric (`species_richness`).
2. A cross-domain slice (heat-stress × rainfall anomaly).
3. A comparative breakdown (top H3 cells by total observations).

The metrics helper is `databox.orchestration.metrics.resolve_metric_query`.
It rewrites `METRIC(<name>)` references and the `__semantic.__table`
placeholder into executable SQL. The single source of truth for the metric
definitions is `transforms/main/metrics/flagship.sql`.

If the warehouse does not yet have the flagship mart populated (fresh
clone, no `task full-refresh` yet), the notebook falls back to a seeded
synthetic DataFrame so the rendered output still has charts.

In [ ]:
from __future__ import annotations

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from databox.config.settings import settings

plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = ["#2a9d8f", "#e76f51", "#264653", "#e9c46a", "#8ab17d"]

print(f"backend={settings.backend}  database={settings.database_path}")

## Connect + load

`settings.database_path` resolves to `data/databox.duckdb` on `local` and
`md:databox` on `motherduck`. Same code path works on both backends.

In [ ]:
def load_flagship() -> pd.DataFrame:
    """Return flagship-mart rows, or a seeded synthetic DataFrame if empty."""
    try:
        conn = duckdb.connect(settings.database_path, read_only=True)
        df = conn.execute(
            "SELECT obs_date, species_code, h3_cell, n_observations, n_checklists, "
            "is_hot_day, prcp_mm_z_7d, discharge_cfs_z_7d "
            "FROM analytics.fct_species_environment_daily"
        ).df()
        conn.close()
        if len(df) > 0:
            df["obs_date"] = pd.to_datetime(df["obs_date"])
            return df
    except Exception as exc:
        print(f"warehouse unavailable ({exc.__class__.__name__}); using synthetic data")

    rng = np.random.default_rng(seed=20260421)
    dates = pd.date_range("2026-01-01", "2026-04-20", freq="D")
    species = [f"sp{i:03d}" for i in range(40)]
    cells = [f"882a1072{i:02x}fffff" for i in range(12)]
    rows = []
    for d in dates:
        day_species = rng.choice(species, size=rng.integers(8, 25), replace=False)
        for sp in day_species:
            rows.append(
                {
                    "obs_date": d,
                    "species_code": sp,
                    "h3_cell": rng.choice(cells),
                    "n_observations": int(rng.integers(1, 60)),
                    "n_checklists": int(rng.integers(1, 15)),
                    "is_hot_day": int(rng.random() < 0.15 + 0.3 * ((d.month - 1) / 11)),
                    "prcp_mm_z_7d": float(rng.normal(0, 1)),
                    "discharge_cfs_z_7d": float(rng.normal(0, 1)),
                }
            )
    return pd.DataFrame(rows)


df = load_flagship()
print(f"rows={len(df):,}  species={df['species_code'].nunique()}  cells={df['h3_cell'].nunique()}")
df.head()

## Metric 1 — daily species richness

`METRIC(species_richness)` is `COUNT(DISTINCT species_code)`. Grouping by
`obs_date` gives platform-wide daily diversity.

In [ ]:
daily = df.groupby("obs_date").agg(species_richness=("species_code", "nunique")).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily["obs_date"], daily["species_richness"], color=PALETTE[0], linewidth=1.6)
ax.set_title("Daily species richness", fontsize=13, loc="left")
ax.set_xlabel("Observation date")
ax.set_ylabel("Distinct species")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Metric 2 — heat stress × rainfall anomaly

Cross-domain slice: bucket days by precipitation z-score and show the
fraction that were hot days. Hotter-drier days should cluster in the
top-right if the hypothesis holds.

In [ ]:
daily_env = (
    df.groupby("obs_date")
    .agg(
        heat_stress_index=("is_hot_day", "mean"),
        prcp_z=("prcp_mm_z_7d", "mean"),
    )
    .reset_index()
)
daily_env["prcp_bucket"] = pd.cut(
    daily_env["prcp_z"],
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["dry", "normal", "wet"],
)
grouped = daily_env.groupby("prcp_bucket", observed=True)["heat_stress_index"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
grouped.plot(kind="bar", ax=ax, color=PALETTE[:3])
ax.set_title("Heat stress by 7-day rainfall bucket", fontsize=13, loc="left")
ax.set_xlabel("7-day rainfall z-score bucket")
ax.set_ylabel("Fraction of days with tmax ≥ 30°C")
ax.set_xticklabels(grouped.index, rotation=0)
plt.tight_layout()
plt.show()

## Metric 3 — top H3 cells by total observations

Comparative breakdown. `METRIC(total_observations)` aggregates across the
grouped slice; ranking H3 cells reveals where bird activity concentrates.

In [ ]:
top_cells = (
    df.groupby("h3_cell")
    .agg(total_observations=("n_observations", "sum"))
    .sort_values("total_observations", ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(
    top_cells["h3_cell"].str[:10] + "…",
    top_cells["total_observations"],
    color=PALETTE[3],
)
ax.invert_yaxis()
ax.set_title("Top 10 H3 cells by total observations", fontsize=13, loc="left")
ax.set_xlabel("Observations (sum)")
ax.set_ylabel("H3 cell")
plt.tight_layout()
plt.show()

## Next steps

- Metric definitions live in
  [`transforms/main/metrics/flagship.sql`](../transforms/main/metrics/flagship.sql).
- The `resolve_metric_query` helper documentation is in
  [`docs/metrics.md`](../docs/metrics.md).
- The reasoning behind SQLMesh metrics is in
  [ADR-0002](../docs/adr/0002-sqlmesh-over-dbt.md).

To use `METRIC(...)` directly against the warehouse:

```python
from databox.orchestration.metrics import resolve_metric_query

sql = resolve_metric_query(
    "SELECT obs_date, METRIC(species_richness) AS sr "
    "FROM __semantic.__table GROUP BY obs_date"
)
df = duckdb.connect(settings.database_path, read_only=True).execute(sql).df()
```